## Aligning NMR spectra with WNetAlign

The following tutorials shows simple example of how to use `wnetalign` package to align 15N-1H HSQC NMR spectra of GB1 protein measured in different temperatures. 

In [1]:
import numpy as np
import pandas as pd
from glob import glob

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATA_PATH = "15N_HSQC_GB1_reduced"

# parameters used in the analysis
MAX_PEAK_FRACTION = 0.1
MAX_DISTANCE = 0.05
TRASH_COST = 0.09

### Loading data 

In our case the spectra are provided as `csv` files where `15N` and `1H` columns contain chemical shifts and column `i` consists intensity of the signal. In this case we scale `15N` dimension by dividing the shift by $10$ to account for its larger chemical shift scale. Moreover, we will include only the signals with the intensity greater than $0.1 \cdot i_{\text{max}}$, where $i_{\text{max}}$ is maximal signal intensity in the spectrum. 

For loading NMR spectra users can use the functions provided in `publication/nmr/nmr/load_spectra.py` or use the simplified function provided below.  

In [10]:
from wnetalign.spectrum import Spectrum

In [11]:
def load_spectrum_basic(path, 
                  dim = 2,
                  scale_nucl={}, #example {'15N':10} - 2D, {'C':10} - 4D
                  max_peak_fraction=0.1,
                  verbose=False,
                  ):
    label = path.split("/")[-1].split(".")[0].split("_")[-1]
    df = pd.read_csv(path)
    if verbose:
        s = df.i.sum()
        print("{}\nnumber of peaks: {}, total signal: {}".format(label, df.shape[0], round(s, 2)))
    df = df[df['i'] > df.i.max()*max_peak_fraction]
    if verbose: 
        print("Peaks with intensities higher than max_intensity * {}".format(max_peak_fraction))
        print("number of peaks: {}, max_intensity * max_peak_fraction: {}, % of signal left: {}".format(df.shape[0], round(df.i.max()*max_peak_fraction, 3), round(100*df.i.sum()/s, 2)))

    # scale specific chemical shifts
    for nucl, scale in scale_nucl.items():
        for c in df.columns:
            if c.startswith(nucl): df[c] = df[c]/scale

    positions = df[df.columns[:dim]].T.to_numpy()
    intensities = df['i'].to_numpy()

    return Spectrum(positions = positions,
                        intensities = intensities,
                        label = label,
                        )

In [21]:
# load first spectrum
S1 = load_spectrum_basic(
    path = f"{DATA_PATH}/gNhsqc_GB1_25C.csv",
    dim = 2,
    scale_nucl = {'15N':10},
    max_peak_fraction=MAX_PEAK_FRACTION,
    verbose=True,
)
print()

# load second spectrum
S2 = load_spectrum_basic(
    path = f"{DATA_PATH}/gNhsqc_GB1_30C.csv",
    dim = 2,
    scale_nucl = {'15N':10},
    max_peak_fraction=MAX_PEAK_FRACTION,
    verbose=True,
)



25C
number of peaks: 68473, total signal: 134.45
Peaks with intensities higher than max_intensity * 0.1
number of peaks: 1431, max_intensity * max_peak_fraction: 0.019, % of signal left: 60.62

30C
number of peaks: 73084, total signal: 120.11
Peaks with intensities higher than max_intensity * 0.1
number of peaks: 1326, max_intensity * max_peak_fraction: 0.016, % of signal left: 55.89


We can access the spectrum label, chemical shifts and intesities in the following way

In [22]:
print("label:", S1.label)
print("chemical shifts:\n", S1.positions)
print("intensities:\n", S1.intensities)
print("sum of intensities: {:.2f}".format(sum(S1.intensities)))

label: 25C
chemical shifts:
 [[10.297   10.3037  10.3037  ... 13.4349  13.4416  13.4416 ]
 [ 6.88056  6.88056  6.90013 ...  7.68286  7.66329  7.68286]]
intensities:
 [0.0262858 0.0418359 0.0205998 ... 0.0406096 0.0196429 0.0237377]
sum of intensities: 81.50


We normalize spectra before the alignment using `Spectrum.normalize()`

In [23]:
S1 = S1.normalized()
S2 = S2.normalized()

In [24]:
print("label:", S1.label)
print("intensities:\n", S1.intensities)
print("sum of intensities: {:.2f}".format(sum(S1.intensities)))

label: 25C
intensities:
 [0.00032253 0.00051333 0.00025276 ... 0.00049828 0.00024102 0.00029126]
sum of intensities: 1.00


## Align spectra

Now, we will align two spectra which can be done using function `align_pair` from `publication/nmr/nmr/align.py` or the code block below. 

We perform the alignment using the `max_distance = 0.05` ($\text{d}_{\text{max}}$ in publication) and `trash_cost = 0.09` ($\delta_{\text{max}}$ in publication). We will perform alignment with automatic intensity scaling (`scale_factor = None`) and using $L2$ metric as a distance (`distance = DistanceMetric.L2`).


In [25]:
from wnetalign.aligner import WNetAligner
from wnet.wnet_cpp import DistanceMetric

In [ ]:
# construct the graph 
solver = WNetAligner(
            empirical_spectrum = S1,
            theoretical_spectra = (S2,), 
            distance = DistanceMetric.L2,
            max_distance = MAX_DISTANCE,
            trash_cost = TRASH_COST,
            scale_factor = None,
            )

In [27]:
solver.set_point([1]) # compute the matching 

We can get the flow cost with `WNetAligner.total_cost()`

In [28]:
print(f"Total cost: {solver.total_cost()}\n")

Total cost: 0.03669994607594704



The transport plan can be retrieved with `WNetAligner.flows()`.

In [29]:
solver.flows()

[Flow(empirical_peak_idx=array([   1,    2,    4, ..., 1429, 1429, 1430],
       shape=(2164,), dtype=int32), theoretical_peak_idx=array([   0,    1,    2, ..., 1323, 1325, 1325],
       shape=(2164,), dtype=int32), flow=array([3.28793842e-04, 2.52759643e-04, 4.81207296e-04, ...,
        1.75924413e-04, 6.50941394e-05, 2.91261636e-04], shape=(2164,)))]

We can save the alignment to csv file using `save_alignment` function provided in `publication/nmr/nmr/align.py` or using the code below.

In [ ]:
res = solver.flows()[0] # get the first (and only) result of the flow computation

scale_nucl={'15N':10}
nuclei=['15N', '1H']

l1 = S1.label
l2 = S2.label

df = pd.DataFrame()
df["empirical_peak_idx"] = res.empirical_peak_idx
df["theoretical_peak_idx"] = res.theoretical_peak_idx
df["flow"] = res.flow

# empirical spectrum / first spectrum
for nucl, pos in zip(nuclei, S1.positions[:, res.empirical_peak_idx]):
    if nucl in scale_nucl: df[f"{l1}_{nucl}"] = pos*scale_nucl[nucl]
    else: df[f"{l1}_{nucl}"] = pos
# theorethical spectrum / second spectrum
for nucl, pos in zip(nuclei, S2.positions[:, res.theoretical_peak_idx]):
    if nucl in scale_nucl: df[f"{l2}_{nucl}"] = pos*scale_nucl[nucl]
    else: df[f"{l2}_{nucl}"] = pos

# save the results
df.to_csv(f"./{l1}_vs_{l2}_{MAX_DISTANCE}_{TRASH_COST}_{solver.total_cost():.4f}.csv", index=False)

In [33]:
df

,empirical_peak_idx,theoretical_peak_idx,flow,25C_15N,25C_1H,30C_15N,30C_1H
0,1,0,0.000329,103.037,6.88056,103.037,6.91970
1,2,1,0.000253,103.037,6.90013,103.037,6.93927
2,4,2,0.000481,103.105,6.88056,103.105,6.91970
3,5,3,0.000359,103.105,6.90013,103.105,6.93927
4,7,4,0.000629,103.172,6.88056,103.172,6.91970
...,...,...,...,...,...,...,...
2159,1428,1323,0.000022,134.349,7.68286,134.281,7.70242
2160,1428,1324,0.000407,134.349,7.68286,134.281,7.72199
2161,1429,1323,0.000176,134.416,7.66329,134.281,7.70242
2162,1429,1325,0.000065,134.416,7.66329,134.349,7.70242
